In [1]:
# Load packages and define project paths
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import os
import time
import matplotlib.colors as mcolors
from pyscenic.aucell import aucell
import gseapy as gp
from gseapy import SingleSampleGSEA
from collections import namedtuple
from scipy import sparse

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RESULT_DIR = PROJECT_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)


c:\Users\23671\AppData\Local\Programs\Python\Python310\lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
c:\Users\23671\AppData\Local\Programs\Python\Python310\lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
c:\Users\23671\AppData\Local\Programs\Python\Python310\lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
c:\Users\23671\AppData\Local\Programs\Python\Python310\lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
c:\Users\23671\AppData\Local\Programs\Python\Python310\lib\site-packag

In [2]:
# Read the four-tissue AnnData object
file_path = DATA_DIR / "GSE201333_RAW" / "GSM6058681_TabulaSapiens.h5ad" / "key_4tissue_celltype_adata.h5ad"
adata = sc.read_h5ad(str(file_path))


In [20]:
# Inspect the AnnData object and cell-type composition
print(adata)
print(adata.shape)
print(adata.obs['organ_tissue'].value_counts())
print(adata.obs['tissue_celltype'].value_counts())


AnnData object with n_obs × n_vars = 86209 × 18985
    obs: 'organ_tissue', 'method', 'donor', 'anatomical_information', 'n_counts_UMIs', 'n_genes', 'cell_ontology_class', 'free_annotation', 'manually_annotated', 'compartment', 'gender', 'tissue_celltype', 'celltype_integration'
    var: 'gene_symbol', 'feature_type', 'ensemblid', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: '_scvi', '_training_mode', 'dendrogram_cell_type_tissue', 'dendrogram_computational_compartment_assignment', 'dendrogram_consensus_prediction', 'dendrogram_tissue_cell_type', 'donor_colors', 'donor_method_colors', 'hvg', 'method_colors', 'neighbors', 'organ_tissue_colors', 'sex_colors', 'tissue_colors', 'umap', 'rank_genes_groups'
    obsm: 'X_pca', 'X_scvi', 'X_scvi_umap', 'X_umap'
    layers: 'decontXcounts', 'raw_counts'
    obsp: 'connectivities', 'distances'
(86209, 18985)
organ_tissue
Blood          49912
Vasculature    15840
Heart          11431
Skin            9026
N

In [12]:
# Add integrated cell-type labels
celltype_mapping = {
    # Blood
    "Blood_cd4-positive, alpha-beta memory t cell": "Blood_Lymphoid_cell",
    "Blood_cd8-positive, alpha-beta cytokine secreting effector t cell": "Blood_Lymphoid_cell",
    "Blood_cd8-positive, alpha-beta t cell": "Blood_Lymphoid_cell",
    "Blood_naive thymus-derived cd4-positive, alpha-beta t cell": "Blood_Lymphoid_cell",
    "Blood_cd4-positive, alpha-beta t cell": "Blood_Lymphoid_cell",
    "Blood_naive b cell": "Blood_Lymphoid_cell",
    "Blood_memory b cell": "Blood_Lymphoid_cell",
    "Blood_plasma cell": "Blood_Lymphoid_cell",
    "Blood_nk cell": "Blood_Lymphoid_cell",
    "Blood_type i nk t cell": "Blood_Lymphoid_cell",
    # Blood myeloid
    "Blood_classical monocyte": "Blood_Myeloid_cell",
    "Blood_monocyte": "Blood_Myeloid_cell",
    "Blood_macrophage": "Blood_Myeloid_cell",
    "Blood_neutrophil": "Blood_Myeloid_cell",
    # Blood non-immune
    "Blood_erythrocyte": "Blood_Erythrocyte",
    "Blood_platelet": "Blood_Platelet",
    
    # Skin
    # Skin lymphoid
    "Skin_t cell": "Skin_Lymphoid_cell",
    "Skin_cd8-positive, alpha-beta memory t cell": "Skin_Lymphoid_cell",
    "Skin_cd8-positive, alpha-beta cytotoxic t cell": "Skin_Lymphoid_cell",
    "Skin_cd4-positive, alpha-beta memory t cell": "Skin_Lymphoid_cell",
    "Skin_regulatory t cell": "Skin_Lymphoid_cell",
    "Skin_nk cell": "Skin_Lymphoid_cell",
    "Skin_nkt cell": "Skin_Lymphoid_cell",
    # Skin myeloid
    "Skin_macrophage": "Skin_Myeloid_cell",
    "Skin_cd1c-positive myeloid dendritic cell": "Skin_Myeloid_cell",
    "Skin_mast cell": "Skin_Myeloid_cell",
    # Skin non-immune
    "Skin_stromal cell": "Skin_Stromal_cell",
    "Skin_muscle cell": "Skin_Muscle_cell",
    "Skin_endothelial cell": "Skin_Endothelial_cell",
    "Skin_epithelial cell": "Skin_Epithelial_cell",
    
    # Heart
    "Heart_cardiac endothelial cell": "Heart_Endothelial_cell",
    "Heart_cardiac muscle cell": "Heart_Muscle_cell",
    "Heart_smooth muscle cell": "Heart_Muscle_cell",
    "Heart_fibroblast of cardiac tissue": "Heart_Fibroblast",
    "Heart_hepatocyte": "Heart_Hepatocyte",

    # Vasculature
    # Vasculature lymphoid (T/NK)
    "Vasculature_t cell": "Vasculature_Lymphoid_cell",
    "Vasculature_nk cell": "Vasculature_Lymphoid_cell",
    # Vasculature myeloid
    "Vasculature_macrophage": "Vasculature_Myeloid_cell",
    "Vasculature_mast cell": "Vasculature_Myeloid_cell",
    # Vasculature non-immune
    "Vasculature_fibroblast": "Vasculature_Fibroblast",
    "Vasculature_smooth muscle cell": "Vasculature_Muscle_cell",
    "Vasculature_pericyte cell": "Vasculature_Pericyte_cell",
    "Vasculature_artery endothelial cell": "Vasculature_Endothelial_cell",
    "Vasculature_endothelial cell": "Vasculature_Endothelial_cell"
}

def add_integrated_celltype(adata):
    """Add the celltype_integration column using the predefined mapping."""
    # Validate required metadata
    if 'tissue_celltype' not in adata.obs.columns:
        raise ValueError("adata.obs does not contain the 'tissue_celltype' column")
    
    # Add integrated labels
    adata.obs['celltype_integration'] = adata.obs['tissue_celltype'].map(celltype_mapping)
    
    # Report unmapped labels
    unmapped = adata.obs[adata.obs['celltype_integration'].isna()]['tissue_celltype'].unique()
    if len(unmapped) > 0:
        print("Warning: the following cell types were not found in the mapping table:")
        print(unmapped)
    
    return adata

# Apply mapping
adata = add_integrated_celltype(adata)


In [13]:
# Check integrated cell-type labels
print(adata.obs['celltype_integration'].value_counts())
len(adata.obs['celltype_integration'].value_counts())


celltype_integration
Blood_Myeloid_cell              25103
Blood_Lymphoid_cell             14086
Blood_Erythrocyte               10484
Heart_Muscle_cell                7427
Vasculature_Fibroblast           5867
Vasculature_Myeloid_cell         3406
Vasculature_Muscle_cell          3075
Skin_Stromal_cell                2943
Heart_Endothelial_cell           2665
Skin_Lymphoid_cell               2575
Vasculature_Endothelial_cell     1702
Skin_Myeloid_cell                1656
Skin_Endothelial_cell            1309
Vasculature_Pericyte_cell        1193
Heart_Hepatocyte                 1089
Vasculature_Lymphoid_cell         597
Skin_Muscle_cell                  437
Heart_Fibroblast                  250
Blood_Platelet                    239
Skin_Epithelial_cell              106
Name: count, dtype: int64


20

In [14]:
# Identify top marker genes for integrated cell types
sc.tl.rank_genes_groups(
    adata,
    groupby='celltype_integration',
    method='wilcoxon',
    n_genes=adata.shape[1],
    pts=True,
    use_raw=False
)

marker_lists = []

# Select positive markers with sufficient detection rate
for tissue in adata.obs['celltype_integration'].cat.categories:
    print(f"Processing cell type: {tissue} …")
    start = time.time()

    df = sc.get.rank_genes_groups_df(adata, group=tissue)

    pct_cols = [
        col for col in df.columns
        if col.lower().startswith('pct') and 'group' in col.lower()
    ]
    if not pct_cols:
        raise KeyError(
            "Cannot find a column indicating expression proportion in group; "
            f"available columns: {df.columns.tolist()}"
        )
    pct_col = pct_cols[0]

    df_filt = df[
        (df['logfoldchanges'] >= 0.25) &
        (df[pct_col] >= 0.25)
    ]

    top100 = (
        df_filt
        .sort_values('logfoldchanges', ascending=False)
        .head(100)['names']
        .tolist()
    )
    marker_lists.extend(top100)

    elapsed = time.time() - start
    print(f" → {tissue}: selected {len(top100)} markers in {elapsed:.1f}s\n")

# Save the unique marker list
unique_markers = list(dict.fromkeys(marker_lists))

pd.Series(unique_markers).to_csv(
    RESULT_DIR / '4tissue_celltype_integration_marker_top100.txt',
    index=False,
    header=False
)
print(f"Total unique markers saved: {len(unique_markers)}")


Processing cell type: Blood_Erythrocyte …
 → Blood_Erythrocyte: selected 100 markers in 0.0s

Processing cell type: Blood_Lymphoid_cell …
 → Blood_Lymphoid_cell: selected 100 markers in 0.1s

Processing cell type: Blood_Myeloid_cell …
 → Blood_Myeloid_cell: selected 100 markers in 0.1s

Processing cell type: Blood_Platelet …
 → Blood_Platelet: selected 100 markers in 0.1s

Processing cell type: Heart_Endothelial_cell …
 → Heart_Endothelial_cell: selected 100 markers in 0.1s

Processing cell type: Heart_Fibroblast …
 → Heart_Fibroblast: selected 100 markers in 0.1s

Processing cell type: Heart_Hepatocyte …
 → Heart_Hepatocyte: selected 100 markers in 0.1s

Processing cell type: Heart_Muscle_cell …
 → Heart_Muscle_cell: selected 100 markers in 0.1s

Processing cell type: Skin_Endothelial_cell …
 → Skin_Endothelial_cell: selected 100 markers in 0.1s

Processing cell type: Skin_Epithelial_cell …
 → Skin_Epithelial_cell: selected 100 markers in 0.1s

Processing cell type: Skin_Lymphoid_cell

In [16]:
# Save the count matrix for BayesPrism
raw = adata.layers['raw_counts']
sparse.issparse(raw)
dense_counts = raw.toarray()

df_counts = pd.DataFrame(
    dense_counts,
    index=adata.obs_names,
    columns=adata.var_names
)

df_counts.to_feather(RESULT_DIR / "key_celltype_4tissue_counts_Deconvolution.feather")


In [17]:
# Save integrated cell-type labels
tissue_celltype_label = adata.obs['celltype_integration'].astype(str).values
np.savetxt(RESULT_DIR / "key_4tissue_celltype_label.txt", tissue_celltype_label, fmt="%s")


In [18]:
# Save the integrated AnnData object
adata.write(RESULT_DIR / "key_4tissue_celltype_adata.h5ad")
